# 02b - Data preparation

**Aim:** prepare non-redundant positive and negative datasets for signal-peptide prediction.

We use the FASTA files from 02a to cluster proteins and retain one representative per cluster. We then recover their metadata from the collected TSV files (slides 27-29). We split the representatives into training and benchmarking datasets; five-fold cross-validation is the next step.

**Materials**
- `02b-DataPreparation-2026.pdf` (slides 4, 6 and 27-29)


## 0. Load the collected datasets

We load the positive and negative FASTA files from 02a. The TSV metadata will be read after clustering.


### 0.1 Imports and paths


In [8]:
import csv
from pathlib import Path

input_dir = Path("../data/collected")


### 0.2 Sequence reader

We keep full sequences in a dictionary keyed by `accession`.


In [9]:
def read_fasta(fasta_path):
    sequences = {}
    with open(fasta_path, encoding="utf-8") as fasta_in:
        for line in fasta_in:
            line = line.strip()
            if line.startswith(">"):
                accession = line[1:].split()[0]
                sequences[accession] = ""
            elif line:
                sequences[accession] += line
    return sequences


### 0.3 Load the collected proteins


In [10]:
positive_sequences = read_fasta(input_dir / "positive_dataset.fasta")
negative_sequences = read_fasta(input_dir / "negative_dataset.fasta")

print("Positive proteins:", len(positive_sequences))
print("Negative proteins:", len(negative_sequences))


Positive proteins: 2957
Negative proteins: 20974


## 1. Reduce redundancy

We cluster positives and negatives separately with MMseqs2, using connected components, 30% minimum identity and 40% coverage of both sequences (slides 27 and 29).

Optional installation with Conda in a Linux notebook environment. Uncomment the cell once, then comment it again for normal runs.


In [11]:
# %conda install -c conda-forge -c bioconda mmseqs2 -y

### 1.1 Cluster the sequences

We use `easy-cluster` with the slide parameters. Each run writes a representative FASTA and a cluster-assignment TSV to `data/prepared/`.


In [12]:
output_dir = Path("../data/prepared")
output_dir.mkdir(parents=True, exist_ok=True)

!mmseqs easy-cluster {input_dir}/positive_dataset.fasta {output_dir}/positive {output_dir}/tmp_positive --min-seq-id 0.3 -c 0.4 --cov-mode 0 --cluster-mode 1
!mmseqs easy-cluster {input_dir}/negative_dataset.fasta {output_dir}/negative {output_dir}/tmp_negative --min-seq-id 0.3 -c 0.4 --cov-mode 0 --cluster-mode 1


easy-cluster ../data/collected/positive_dataset.fasta ../data/prepared/positive ../data/prepared/tmp_positive --min-seq-id 0.3 -c 0.4 --cov-mode 0 --cluster-mode 1 

MMseqs Version:                     	18.8cc5c
Substitution matrix                 	aa:blosum62.out,nucl:nucleotide.out
Seed substitution matrix            	aa:VTML80.out,nucl:nucleotide.out
Sensitivity                         	4
k-mer length                        	0
Target search mode                  	0
k-score                             	seq:2147483647,prof:2147483647
Alphabet size                       	aa:21,nucl:5
Max sequence length                 	65535
Max results per query               	20
Split database                      	0
Split mode                          	2
Split memory limit                  	0
Coverage threshold                  	0.4
Coverage mode                       	0
Compositional bias                  	1
Compositional bias scale            	1
Diagonal scoring                    	true
Exact k-m

### 1.2 Select representatives

The `*_rep_seq.fasta` files contain one representative per cluster (slide 28).


In [13]:
positive_representatives = read_fasta(output_dir / "positive_rep_seq.fasta")
negative_representatives = read_fasta(output_dir / "negative_rep_seq.fasta")

print("Positive representatives:", len(positive_representatives))
print("Negative representatives:", len(negative_representatives))


Positive representatives: 1102
Negative representatives: 9082


### 1.3 Rebuild the metadata files

We read the TSV files from 02a and retain only the representatives, preserving the original columns (slide 29). This selects rows by accession without repeating the collection filters.


In [14]:
def read_metadata(tsv_path):
    metadata = {}
    with open(tsv_path, encoding="utf-8", newline="") as tsv_in:
        for row in csv.DictReader(tsv_in, delimiter="\t"):
            row["length"] = int(row["length"])
            if "cleavage_pos" in row:
                row["cleavage_pos"] = int(row["cleavage_pos"])
            if "has_tm_in_90" in row:
                row["has_tm_in_90"] = row["has_tm_in_90"] == "True"
            metadata[row["accession"]] = row
    return metadata



In [15]:
positive_metadata = read_metadata(input_dir / "positive_dataset.tsv")
negative_metadata = read_metadata(input_dir / "negative_dataset.tsv")


def write_representative_metadata(metadata, representatives, tsv_path):
    columns = list(next(iter(metadata.values())))
    with open(tsv_path, "w", encoding="utf-8", newline="") as tsv_out:
        writer = csv.DictWriter(tsv_out, fieldnames=columns, delimiter="\t")
        writer.writeheader()
        for accession in representatives:
            writer.writerow(metadata[accession])


write_representative_metadata(
    positive_metadata, positive_representatives, output_dir / "positive_dataset.tsv"
)
write_representative_metadata(
    negative_metadata, negative_representatives, output_dir / "negative_dataset.tsv"
)


## 2. Split training and benchmarking datasets

We randomly assign 80% of the representatives in each class to training and the remaining 20% to benchmarking (slides 4, 6 and 29). We use seed 42 to make the split reproducible and round the training count down to a whole number.

In [16]:
import random

random_seed = 42
training_fraction = 0.8
rng = random.Random(random_seed)


def split_representatives(representatives, training_fraction, rng):
    accessions = list(representatives)
    rng.shuffle(accessions)
    n_training = int(training_fraction * len(accessions))
    return accessions[:n_training], accessions[n_training:]


positive_training, positive_benchmarking = split_representatives(
    positive_representatives, training_fraction, rng
)
negative_training, negative_benchmarking = split_representatives(
    negative_representatives, training_fraction, rng
)

print("Training positives:", len(positive_training))
print("Training negatives:", len(negative_training))
print("Benchmarking positives:", len(positive_benchmarking))
print("Benchmarking negatives:", len(negative_benchmarking))

Training positives: 881
Training negatives: 7265
Benchmarking positives: 221
Benchmarking negatives: 1817


### 2.1 Save the datasets

We save each class as FASTA and TSV in `data/prepared/training/` and `data/prepared/benchmarking/`, keeping the original metadata columns and the same accession order in both formats.

In [17]:
def write_split(sequences, metadata, accessions, output_prefix):
    write_representative_metadata(metadata, accessions, output_prefix.with_suffix(".tsv"))
    with open(output_prefix.with_suffix(".fasta"), "w", encoding="utf-8") as fasta_out:
        for accession in accessions:
            print(f">{accession}", sequences[accession], sep="\n", file=fasta_out)


training_dir = output_dir / "training"
benchmarking_dir = output_dir / "benchmarking"
training_dir.mkdir(parents=True, exist_ok=True)
benchmarking_dir.mkdir(parents=True, exist_ok=True)

write_split(positive_representatives, positive_metadata, positive_training,
            training_dir / "positive_dataset")
write_split(negative_representatives, negative_metadata, negative_training,
            training_dir / "negative_dataset")
write_split(positive_representatives, positive_metadata, positive_benchmarking,
            benchmarking_dir / "positive_dataset")
write_split(negative_representatives, negative_metadata, negative_benchmarking,
            benchmarking_dir / "negative_dataset")